# Hugging Face TTS Experiment

This notebook loads a free Hugging Face text-to-speech model, generates an insurance-call prompt, saves the result as a WAV file, and plays it back.

The first run downloads model weights. Use synthetic text only while experimenting; do not put customer data into this notebook.

## 1. Install and Import Dependencies

Run this cell once for the notebook's selected Python environment. Kokoro downloads its model files from Hugging Face on first use.

In [ ]:
%pip install -q kokoro torch scipy soundfile matplotlib

In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy
import soundfile as sf
import torch
from IPython.display import Audio, display
from kokoro import KPipeline

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"SciPy version: {scipy.__version__}")

c:\dev\telemer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
SciPy version: 1.18.1


## 2. Select and Load a Hugging Face TTS Model

Kokoro provides Hindi voices, but the standard Kokoro-82M release does not include a dedicated Indian-English voice. This experiment uses a Hindi voice with English text as the closest available Indian-region approximation. For a true Indian-English accent, compare a model trained specifically on Indian English later.

In [9]:
MODEL_ID = "hexgrad/Kokoro-82M"
LANGUAGE_CODE = "h"  # Hindi phonemizer; closest Kokoro option for an Indian-region voice
VOICE_OPTIONS = ["hf_alpha", "hf_beta", "hm_omega", "hm_psi"]
VOICE = VOICE_OPTIONS[0]
SPEED = 1.0
SAMPLING_RATE = 24_000

kokoro_pipeline = KPipeline(
    lang_code=LANGUAGE_CODE,
    repo_id=MODEL_ID,
    device=str(DEVICE),
)

print(f"Loaded {MODEL_ID}")
print(f"Voice: {VOICE}; available Hindi voices: {VOICE_OPTIONS}")
print(f"Speed: {SPEED}x; sampling rate: {SAMPLING_RATE:,} Hz")

c:\dev\telemer\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\dev\telemer\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loaded hexgrad/Kokoro-82M
Voice: hf_alpha; available Hindi voices: ['hf_alpha', 'hf_beta', 'hm_omega', 'hm_psi']
Speed: 1.0x; sampling rate: 24,000 Hz


## 3. Prepare Input Text

Use synthetic insurance-call text while testing. English text is passed through Kokoro's Hindi pipeline as an accent experiment; compare the available Hindi voices before choosing one for teleMER.

In [19]:
text_samples = [
    "Hello, this is teleMER calling about your insurance application.",
    "To continue, can you tell me whether you have had any prior claims?",
    "हिंदी पाठ्यपुस्तकें विभिन्न शिक्षा बोर्डों और कक्षाओं के अनुसार उपलब्ध हैं। प्रमुख स्रोत और जानकारी नीचे दी गई है:",
    "ہندی پاٹھے پوستھ کے ویبھین نشکشا بورو اور ققشاؤ کے انوصار اپلاب دھے پرموک صروط اور جانکاری نیچے دی گئی ہے۔"
]

text = text_samples[1]
print(text)

To continue, can you tell me whether you have had any prior claims?


## 4. Generate Speech Audio

Kokoro returns one or more audio chunks from its generator. The chunks are concatenated so the rest of the notebook can treat the result as one waveform.

In [20]:
start_time = time.perf_counter()
audio_chunks = []

for _, _, audio in kokoro_pipeline(
    text,
    voice=VOICE,
    speed=SPEED,
    split_pattern=r"\n+",
):
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    audio_chunks.append(np.asarray(audio, dtype=np.float32))

if not audio_chunks:
    raise RuntimeError("Kokoro generated no audio chunks")

waveform = np.concatenate(audio_chunks)
sampling_rate = SAMPLING_RATE
duration_seconds = len(waveform) / sampling_rate
inference_seconds = time.perf_counter() - start_time

print(f"Sampling rate: {sampling_rate:,} Hz")
print(f"Duration: {duration_seconds:.2f} seconds")
print(f"Inference time: {inference_seconds:.2f} seconds")

Sampling rate: 24,000 Hz
Duration: 4.78 seconds
Inference time: 13.97 seconds


## 5. Save and Play the Audio

The output is saved under `artifacts/` and embedded in the notebook for immediate listening.

In [21]:
output_path = ARTIFACT_DIR / "tts_sample2.wav"
sf.write(output_path, waveform, sampling_rate)

print(f"Saved: {output_path.resolve()}")
display(Audio(waveform, rate=sampling_rate))

Saved: C:\dev\telemer\experiments\artifacts\tts_sample2.wav


## 6. Inspect and Adjust Audio Output

Inspect the waveform and generate a second sample. Kokoro's main controls here are the voice, speed, and input text.

In [ ]:
time_axis = np.arange(len(waveform)) / sampling_rate

fig, axis = plt.subplots(figsize=(14, 3))
axis.plot(time_axis, waveform, linewidth=0.7)
axis.set_title("Generated Speech Waveform")
axis.set_xlabel("Time (seconds)")
axis.set_ylabel("Amplitude")
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print({
    "model": MODEL_ID,
    "device": str(DEVICE),
    "sampling_rate_hz": sampling_rate,
    "samples": len(waveform),
    "duration_seconds": round(duration_seconds, 2),
    "peak_amplitude": round(float(np.max(np.abs(waveform))), 4),
})

In [ ]:
def synthesize(text_to_synthesize: str, filename: str):
    sample_chunks = []
    for _, _, sample_audio in kokoro_pipeline(
        text_to_synthesize,
        voice=VOICE,
        speed=SPEED,
        split_pattern=r"\n+",
    ):
        if isinstance(sample_audio, torch.Tensor):
            sample_audio = sample_audio.detach().cpu().numpy()
        sample_chunks.append(np.asarray(sample_audio, dtype=np.float32))

    if not sample_chunks:
        raise RuntimeError("Kokoro generated no audio chunks")

    sample_waveform = np.concatenate(sample_chunks)
    sample_path = ARTIFACT_DIR / filename
    sf.write(sample_path, sample_waveform, SAMPLING_RATE)
    print(f"Saved: {sample_path.resolve()}")
    display(Audio(sample_waveform, rate=SAMPLING_RATE))

synthesize(
    "Before we continue, do you have any questions about this insurance call?",
    "tts_follow_up.wav",
)